# 11 — Does finance structure add information beyond total dollars? (2024)

Notebook 10 showed that **campaign scale is the main top-line signal**.

This notebook asks a narrower question:

> Once we already know how much a candidate raised or spent, do finer-grained
> finance features explain additional electoral support?

This is a stronger test than simply asking which feature has the largest
standalone correlation.

**Terminology:** the data count contribution **records**, not unique donors.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 100)

# Find the repository root from either the repo root or notebooks/.
cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)

YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


## 2. Load the top-line table and finance profiles

We reuse Notebook 10's candidate table instead of rebuilding the same merge.


In [2]:
topline_path = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "topline"
    / "candidate_topline_finance_support.csv"
)

fundraising_path = (
    fundraising_processed_dir(YEAR, CONTEST)
    / "openelections_candidate_fundraising_profiles_wide.csv"
)

spending_path = (
    spending_processed_dir(YEAR, CONTEST)
    / "orestar_candidate_spending_profiles_wide.csv"
)

analysis = pd.read_csv(topline_path)
fundraising = pd.read_csv(fundraising_path)
spending = pd.read_csv(spending_path)

# Keep only linked spending rows.
spending = spending[
    spending["candidate_key"].notna()
].copy()

if spending["candidate_key"].duplicated().any():
    raise ValueError(
        "More than one spending profile exists for a candidate. "
        "Combine the source profiles upstream before running this notebook."
    )

print("Top-line candidates:", len(analysis))
print("Fundraising profiles:", len(fundraising))
print("Linked spending profiles:", len(spending))


Top-line candidates: 98
Fundraising profiles: 65
Linked spending profiles: 66


In [3]:
# Keep profile variables but avoid duplicating columns already in the top-line table.
fundraising_features = [
    "average_contribution",
    "median_contribution",
    "amount_share_micro",
    "amount_share_small",
    "amount_share_medium",
    "amount_share_large",
    "amount_share_mega",
    "contribution_share_micro",
    "contribution_share_small",
    "contribution_share_medium",
    "contribution_share_large",
    "contribution_share_mega",
]

spending_features = [
    "average_expenditure",
    "median_expenditure",
    "spending_amount_share_micro",
    "spending_amount_share_small",
    "spending_amount_share_medium",
    "spending_amount_share_large",
    "spending_amount_share_mega",
    "spending_count_share_micro",
    "spending_count_share_small",
    "spending_count_share_medium",
    "spending_count_share_large",
    "spending_count_share_mega",
]

fundraising_keep = [
    "candidate_key",
] + [
    column
    for column in fundraising_features
    if column in fundraising.columns
]

spending_keep = [
    "candidate_key",
] + [
    column
    for column in spending_features
    if column in spending.columns
]

analysis = analysis.merge(
    fundraising[fundraising_keep],
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

analysis = analysis.merge(
    spending[spending_keep],
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

analysis["fundraising_thousands"] = (
    analysis["fundraising"]
    / 1000
)

print("Candidates in merged table:", len(analysis))


Candidates in merged table: 98


## 3. First check: which fine-grained variables look interesting alone?

This is only a **screening table**. A strong standalone correlation does not
prove the feature adds information beyond total money.


In [4]:
features = [
    "contribution_count",
    *[
        column
        for column in fundraising_features
        if column in analysis.columns
    ],
    *[
        column
        for column in spending_features
        if column in analysis.columns
    ],
]

rows = []

for feature in features:
    pair = analysis[
        [feature, "mentions"]
    ].dropna()

    if len(pair) < 4:
        continue

    rows.append(
        {
            "feature": feature,
            "n": len(pair),
            "pearson": pair[feature].corr(
                pair["mentions"]
            ),
            "spearman": pair[feature].corr(
                pair["mentions"],
                method="spearman",
            ),
        }
    )

screening = pd.DataFrame(rows)

screening["abs_pearson"] = (
    screening["pearson"].abs()
)

display(
    screening
    .sort_values(
        "abs_pearson",
        ascending=False,
    )
    .head(12)
    .drop(columns="abs_pearson")
    .round(3)
)


,feature,n,pearson,spearman
0,contribution_count,65,0.725,0.755
19,spending_amount_share_mega,65,0.598,0.658
13,average_expenditure,65,0.597,0.656
24,spending_count_share_mega,65,0.586,0.670
18,spending_amount_share_large,65,-0.541,-0.605
17,spending_amount_share_medium,65,-0.463,-0.568
15,spending_amount_share_micro,65,-0.407,-0.557
16,spending_amount_share_small,65,-0.403,-0.556
20,spending_count_share_micro,65,-0.330,-0.421
7,amount_share_mega,65,0.264,0.289


### What the earlier run showed

Contribution count was the strongest standalone fine-grained feature, but it
still did not beat total fundraising on mentions (R² about **0.525** vs
**0.607**). fileciteturn54file0

That does **not** yet answer whether contribution count adds information
*conditional on total fundraising*. We test that next.


## 4. Direct test: does contribution count add anything beyond fundraising?

We compare:

`mentions ~ fundraising`

with

`mentions ~ fundraising + contribution count`

We use **raw fundraising**, because Notebook 10 found that raw dollars fit
fundraising better than log dollars.

The key numbers are:

- adjusted R²;
- the p-value on contribution count.


In [5]:
breadth_rows = []

for outcome in [
    "mentions",
    "first_place_votes",
]:
    data = analysis.dropna(
        subset=[
            "fundraising_thousands",
            "contribution_count",
            outcome,
        ]
    ).copy()

    y = data[outcome].to_numpy(dtype=float)

    # Model 1: money only.
    x_money = sm.add_constant(
        data[
            ["fundraising_thousands"]
        ].to_numpy(dtype=float)
    )

    money_model = sm.OLS(
        y,
        x_money,
    ).fit()

    # Model 2: money + contribution count.
    x_breadth = sm.add_constant(
        data[
            [
                "fundraising_thousands",
                "contribution_count",
            ]
        ].to_numpy(dtype=float)
    )

    breadth_model = sm.OLS(
        y,
        x_breadth,
    ).fit()

    breadth_rows.append(
        {
            "outcome": outcome,
            "n": len(data),
            "money_only_adj_r2": money_model.rsquared_adj,
            "money_plus_count_adj_r2": breadth_model.rsquared_adj,
            "adj_r2_change": (
                breadth_model.rsquared_adj
                - money_model.rsquared_adj
            ),
            "count_coefficient": breadth_model.params[2],
            "count_p_value": breadth_model.pvalues[2],
        }
    )

breadth_test = pd.DataFrame(
    breadth_rows
)

display(
    breadth_test.round(
        {
            "money_only_adj_r2": 3,
            "money_plus_count_adj_r2": 3,
            "adj_r2_change": 3,
            "count_coefficient": 3,
            "count_p_value": 4,
        }
    )
)


,outcome,n,money_only_adj_r2,money_plus_count_adj_r2,adj_r2_change,count_coefficient,count_p_value
0,mentions,65,0.601,0.627,0.026,6.414,0.0225
1,first_place_votes,65,0.558,0.571,0.013,1.831,0.0953


In [6]:
# A plain-English checkpoint generated from the model results.
for row in breadth_test.itertuples(index=False):
    print()
    print("Outcome:", row.outcome)

    if (
        row.count_p_value < 0.05
        and row.adj_r2_change > 0
    ):
        print(
            "Contribution count adds evidence beyond total fundraising "
            "in this specification."
        )
    else:
        print(
            "We do not find clear evidence that contribution count adds "
            "predictive value beyond total fundraising in this specification."
        )



Outcome: mentions
Contribution count adds evidence beyond total fundraising in this specification.

Outcome: first_place_votes
We do not find clear evidence that contribution count adds predictive value beyond total fundraising in this specification.


### Interpretation rule

Do **not** write “breadth does not exist.”

A null result here means something narrower:

> In this candidate-level linear specification, contribution-record count does
> not provide clear additional predictive information once total fundraising
> is already included.

That is still a useful negative result.


## 5. Conditional screen: do profile shares add anything?

Profile shares are compositional: the five shares add to 1. We therefore test
them one at a time and treat the results as exploratory.

For fundraising, the baseline is raw dollars.

For spending, the baseline is log spending because that fit better in
Notebook 10.


In [7]:
analysis["log_spending"] = np.log1p(
    analysis["total_spending"]
)

def conditional_screen(
    data,
    baseline,
    feature_list,
    outcome="mentions",
):
    rows = []

    for feature in feature_list:
        if feature not in data.columns:
            continue

        model_data = data[
            [baseline, feature, outcome]
        ].dropna()

        if len(model_data) < 5:
            continue

        y = model_data[
            outcome
        ].to_numpy(dtype=float)

        x_base = sm.add_constant(
            model_data[
                [baseline]
            ].to_numpy(dtype=float)
        )

        x_plus = sm.add_constant(
            model_data[
                [baseline, feature]
            ].to_numpy(dtype=float)
        )

        base_model = sm.OLS(
            y,
            x_base,
        ).fit()

        plus_model = sm.OLS(
            y,
            x_plus,
        ).fit()

        rows.append(
            {
                "feature": feature,
                "n": len(model_data),
                "baseline_adj_r2": base_model.rsquared_adj,
                "with_feature_adj_r2": plus_model.rsquared_adj,
                "adj_r2_change": (
                    plus_model.rsquared_adj
                    - base_model.rsquared_adj
                ),
                "feature_p_value": plus_model.pvalues[2],
            }
        )

    return pd.DataFrame(rows)


fundraising_screen = conditional_screen(
    analysis,
    baseline="fundraising",
    feature_list=[
        feature
        for feature in fundraising_features
        if feature in analysis.columns
    ],
)

spending_screen = conditional_screen(
    analysis,
    baseline="log_spending",
    feature_list=[
        feature
        for feature in spending_features
        if feature in analysis.columns
    ],
)

print("Fundraising features:")
display(
    fundraising_screen
    .sort_values(
        "adj_r2_change",
        ascending=False,
    )
    .head(10)
    .round(4)
)

print("Spending features:")
display(
    spending_screen
    .sort_values(
        "adj_r2_change",
        ascending=False,
    )
    .head(10)
    .round(4)
)


Fundraising features:


,feature,n,baseline_adj_r2,with_feature_adj_r2,adj_r2_change,feature_p_value
6,amount_share_mega,65,0.6007,0.6251,0.0244,0.0274
11,contribution_share_mega,65,0.6007,0.6120,0.0113,0.0974
5,amount_share_large,65,0.6007,0.6033,0.0026,0.2397
10,contribution_share_large,65,0.6007,0.6021,0.0014,0.2741
7,contribution_share_micro,65,0.6007,0.5994,-0.0013,0.3742
3,amount_share_small,65,0.6007,0.5982,-0.0025,0.4402
2,amount_share_micro,65,0.6007,0.5977,-0.0030,0.4678
0,average_contribution,65,0.6007,0.5977,-0.0030,0.4685
8,contribution_share_small,65,0.6007,0.5972,-0.0035,0.5052
4,amount_share_medium,65,0.6007,0.5959,-0.0048,0.6200


Spending features:


,feature,n,baseline_adj_r2,with_feature_adj_r2,adj_r2_change,feature_p_value
8,spending_count_share_small,65,0.476,0.4948,0.0188,0.0723
3,spending_amount_share_small,65,0.476,0.4900,0.0140,0.1032
0,average_expenditure,65,0.476,0.4848,0.0088,0.1547
10,spending_count_share_large,65,0.476,0.4798,0.0039,0.2301
11,spending_count_share_mega,65,0.476,0.4746,-0.0014,0.3647
4,spending_amount_share_medium,65,0.476,0.4722,-0.0037,0.4598
1,median_expenditure,65,0.476,0.4702,-0.0058,0.5799
5,spending_amount_share_large,65,0.476,0.4701,-0.0058,0.5809
7,spending_count_share_micro,65,0.476,0.4698,-0.0062,0.6094
2,spending_amount_share_micro,65,0.476,0.4697,-0.0062,0.6127


## 6. Interactive zoom: money, contribution count, and mentions

This figure is descriptive. Point size is contribution count; hover gives the
candidate name.

A larger point is **more contribution records**, not necessarily more unique
people.


In [8]:
plot_data = analysis.dropna(
    subset=[
        "fundraising",
        "contribution_count",
        "mentions",
    ]
).copy()

plot_data["district"] = (
    plot_data["district"]
    .astype(str)
)

fig = px.scatter(
    plot_data,
    x="fundraising",
    y="mentions",
    size="contribution_count",
    color="district",
    symbol="is_viable",
    hover_name="canonical_candidate",
    hover_data={
        "fundraising": ":$,.0f",
        "contribution_count": ":,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
    },
    size_max=28,
    log_x=True,
    labels={
        "fundraising": "Fundraising ($, log scale for display)",
        "mentions": "Ballot mentions",
        "contribution_count": "Contribution records",
        "district": "District",
    },
    title="Same money, different contribution counts? Hover to explore",
)

fig.show()


## 7. Conclusion

This notebook should answer **one sentence** after it is rerun:

> Do finer-grained finance measures add meaningful information beyond total
> campaign scale?

The earlier univariate run already showed that no single fine-grained feature
beat total fundraising. fileciteturn54file3

The direct conditional tests above are the stronger evidence. Use those
results — not the old “breadth exists / breadth does not exist” language — in
the memo.

Spending-size shares remain hard to interpret substantively because they
describe transaction size, not spending purpose.


## 8. Export

In [9]:
output_dir = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "fine_grained"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

screening.to_csv(
    output_dir / "fine_grained_screening_correlations.csv",
    index=False,
)

breadth_test.to_csv(
    output_dir / "contribution_count_conditional_test.csv",
    index=False,
)

fundraising_screen.to_csv(
    output_dir / "fundraising_conditional_feature_screen.csv",
    index=False,
)

spending_screen.to_csv(
    output_dir / "spending_conditional_feature_screen.csv",
    index=False,
)

print("SAVED:", output_dir)


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/2024/question_3/fine_grained
